In [4]:
import os
import re
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()

True

In [5]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing")

os.environ["OPENAI_API_KEY"] = api_key

In [6]:
api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is missing")

os.environ["ANTHROPIC_API_KEY"] = api_key

# OPEN AI

In [ ]:
from openai import OpenAI

In [ ]:
def run_gpt55_on_questions(
    input_csv,
    output_csv,
    prompt_col="Prompt",
    model="gpt-5.5",
    question_id_col=None,
    gold_col=None,
    level_col="level",
    medical_specialty_col="medical_specialty",
    max_retries=5,
    save_every=1,
):
    """
    Run GPT-5.5 on medical MCQ prompts and save predictions.

    Expected input CSV columns:
    - Prompt
    - Group
    - Question
    - gold/answer column, e.g. 'gold', 'answer', 'Answer', 'Correct Answer'
    - optional: level
    - optional: medical_specialty

    Output CSV columns:
    - question_id
    - gold
    - prediction
    - raw_output
    - correct
    - level
    - medical_specialty
    - model
    """

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    if prompt_col not in df.columns:
        raise ValueError(f"Missing prompt column: {prompt_col}")


    if gold_col is None:
        raise ValueError(
            "Could not find the gold answer column. "
            "Please pass gold_col='your_column_name'."
        )

    def clean_value(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def normalize_gold(value):
        """
        Extract gold answer letter from values like:
        A
        a
        A.
        Option A
        """
        text = clean_value(value).upper()
        match = re.search(r"\b([A-E])\b", text)
        return match.group(1) if match else text[:1]

    def extract_prediction(raw_output, allowed_letters=None):
        """
        Extract the first valid option letter from the model output.
        Because the prompt asks for only one letter, this is usually enough.
        """

        raw_output = clean_value(raw_output).upper()

        if allowed_letters:
            allowed_letters = [x.upper() for x in allowed_letters]
        else:
            allowed_letters = ["A", "B", "C", "D", "E"]

        # First try exact single-letter output
        if raw_output in allowed_letters:
            return raw_output

        # Then try to find a standalone letter
        pattern = r"\b(" + "|".join(map(re.escape, allowed_letters)) + r")\b"
        match = re.search(pattern, raw_output)
        if match:
            return match.group(1)

        # Fallback: first character if valid
        if raw_output and raw_output[0] in allowed_letters:
            return raw_output[0]

        return ""

    def call_model(prompt, model="gpt-5.5", max_retries=5):
        """
        Calls GPT-5.5 and returns visible text.
        Prevents silently saving empty outputs.
        """

        last_error = None

        for attempt in range(max_retries):
            try:
                response = client.responses.create(
                    model=model,
                    reasoning={"effort": "low"},
                    input=[
                        {
                            "role": "user",
                            "content": prompt,
                        }
                    ],
                    max_output_tokens=512,
                )

                raw_output = response.output_text.strip() if response.output_text else ""

                # Debug info
                if raw_output == "":
                    print("Empty output detected")
                    print("Status:", response.status)
                    print("Incomplete details:", response.incomplete_details)
                    print("Usage:", response.usage)

                    # Retry with more tokens
                    response = client.responses.create(
                        model=model,
                        reasoning={"effort": "low"},
                        input=[
                            {
                                "role": "user",
                                "content": prompt,
                            }
                        ],
                        max_output_tokens=2048,
                    )

                    raw_output = response.output_text.strip() if response.output_text else ""

                if raw_output:
                    return raw_output

                raise ValueError("Model returned empty output even after retry.")

            except Exception as e:
                last_error = e
                wait_time = 2 ** attempt
                print(f"Error: {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)

        raise RuntimeError(f"Failed after {max_retries} retries. Last error: {last_error}")

    
    # Read already processed question IDs if results CSV exists
    processed_ids = set()

    if output_csv.exists():
        old_results = pd.read_csv(output_csv, encoding="utf-8-sig")
        
        if "question_id" in old_results.columns:
            processed_ids = set(old_results["question_id"].astype(str))
        elif "Question_id" in old_results.columns:
            processed_ids = set(old_results["Question_id"].astype(str))

    print(f"Already processed: {len(processed_ids)} questions")

    results = old_results.to_dict("records") if output_csv.exists() else []
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        question_id = (
            clean_value(row[question_id_col])
            if question_id_col is not None
            else str(idx)
        )

        # Skip if already in results CSV
        if str(question_id) in processed_ids:
            continue

        gold = normalize_gold(row[gold_col])

        group = clean_value(row["Group"]).upper() if "Group" in df.columns else "ABCDE"
        allowed_letters = [letter for letter in group if letter in ["A", "B", "C", "D", "E"]]

        prompt = clean_value(row[prompt_col])
        prompt += "\n\nReturn exactly one uppercase letter from the allowed options. No explanation."

        raw_output = call_model(prompt, model=model)
        prediction = extract_prediction(raw_output, allowed_letters)

        correct = int(prediction == gold)

        result = {
            "question_id": question_id,
            "gold": gold,
            "prediction": prediction,
            "raw_output": raw_output,
            "correct": correct,
            "level": clean_value(row[level_col]) if level_col in df.columns else "",
            "medical_specialty": (
                clean_value(row[medical_specialty_col])
                if medical_specialty_col in df.columns
                else ""
            ),
            "model": model,
        }

        results.append(result)

        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame(
        results,
        columns=[
            "question_id",
            "gold",
            "prediction",
            "raw_output",
            "correct",
            "level",
            "medical_specialty",
            "model",
        ],
    )

    results_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    return results_df

In [ ]:
results = run_gpt55_on_questions(
    input_csv="data/cleaned/medarabench_test_with_prompts.csv",
    output_csv="results/gpt55_results.csv",
    prompt_col="Prompt",
    model="gpt-5.5",
    gold_col="Correct Answer",
    question_id_col="Question Number",
    level_col="Level",
    medical_specialty_col="Medical Specialty",
)

Already processed: 20 questions


100%|██████████| 25/25 [00:25<00:00,  1.03s/it]


# CLAUDE

In [7]:
from anthropic import Anthropic

In [ ]:
def run_claude_opus48_on_questions(
    input_csv,
    output_csv,
    prompt_col="Prompt",
    model="claude-opus-4-8",
    question_id_col=None,
    gold_col=None,
    level_col="level",
    medical_specialty_col="medical_specialty",
    max_retries=5,
    save_every=1,
):
    """
    Run Claude Opus 4.8 on medical MCQ prompts and save predictions.

    Output CSV columns:
    - question_id
    - gold
    - prediction
    - raw_output
    - correct
    - level
    - medical_specialty
    - model
    - stop_reason
    - refusal_category
    - refusal_type
    """

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise ValueError(
            "ANTHROPIC_API_KEY is not set. Run:\n"
            "os.environ['ANTHROPIC_API_KEY'] = getpass('Enter ANTHROPIC_API_KEY: ')"
        )

    client = Anthropic(api_key=api_key)

    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    if prompt_col not in df.columns:
        raise ValueError(f"Missing prompt column: {prompt_col}")

    if gold_col is None:
        raise ValueError(
            "Could not find the gold answer column. "
            "Please pass gold_col='your_column_name'."
        )

    output_columns = [
        "question_id",
        "gold",
        "prediction",
        "raw_output",
        "correct",
        "level",
        "medical_specialty",
        "model",
        "stop_reason",
        "refusal_category",
        "refusal_type",
    ]

    def clean_value(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def normalize_gold(value):
        text = clean_value(value).upper()
        match = re.search(r"\b([A-E])\b", text)
        return match.group(1) if match else text[:1]

    def extract_prediction(raw_output, allowed_letters=None):
        raw_output = clean_value(raw_output).upper()

        if allowed_letters:
            allowed_letters = [x.upper() for x in allowed_letters]
        else:
            allowed_letters = ["A", "B", "C", "D", "E","F"]

        if raw_output in allowed_letters:
            return raw_output

        pattern = r"\b(" + "|".join(map(re.escape, allowed_letters)) + r")\b"
        match = re.search(pattern, raw_output)
        if match:
            return match.group(1)

        if raw_output and raw_output[0] in allowed_letters:
            return raw_output[0]

        return ""

    def call_model(prompt, model="claude-opus-4-8", max_retries=5):
        """
        Calls Claude Opus 4.8.

        Returns a dictionary:
        {
            "raw_output": str,
            "stop_reason": str,
            "refusal_category": str or None,
            "refusal_type": str or None,
        }
        """

        system_prompt = (
            "You are evaluating multiple-choice questions from a medical benchmark. "
            "The task is educational and for model benchmarking only. "
            "Select the single best answer option. "
            "Do not provide clinical advice, instructions, procedures, or explanations. "
            "Return only one uppercase letter."
        )

        last_error = None

        for attempt in range(1, max_retries + 1):
            try:
                response = client.messages.create(
                    model=model,
                    max_tokens=16,
                    system=system_prompt,
                    messages=[
                        {
                            "role": "user",
                            "content": prompt,
                        }
                    ],
                    # Do not set temperature/top_p/top_k for Claude Opus 4.8
                )

                stop_reason = getattr(response, "stop_reason", None)
                stop_details = getattr(response, "stop_details", None)

                if stop_reason == "refusal":
                    refusal_category = None
                    refusal_type = None

                    if stop_details is not None:
                        refusal_category = getattr(stop_details, "category", None)
                        refusal_type = getattr(stop_details, "type", None)

                    print(
                        f"Claude refusal detected: "
                        f"category={refusal_category}, type={refusal_type}"
                    )

                    return {
                        "raw_output": "",
                        "stop_reason": "refusal",
                        "refusal_category": refusal_category,
                        "refusal_type": refusal_type,
                    }

                raw_output_parts = []

                for block in response.content:
                    if getattr(block, "type", None) == "text":
                        text = getattr(block, "text", "")
                        if text:
                            raw_output_parts.append(text)

                raw_output = "\n".join(raw_output_parts).strip()

                if raw_output:
                    return {
                        "raw_output": raw_output,
                        "stop_reason": stop_reason,
                        "refusal_category": None,
                        "refusal_type": None,
                    }

                try:
                    print("Empty Claude response:")
                    print(response.model_dump_json(indent=2))
                except Exception:
                    print(response)

                raise RuntimeError(
                    f"Claude returned empty text output. stop_reason={stop_reason}"
                )

            except Exception as e:
                last_error = e

                if attempt == max_retries:
                    break

                wait_time = min(2 ** attempt, 60)
                print(
                    f"API/error attempt {attempt}/{max_retries}: {e}. "
                    f"Retrying in {wait_time}s..."
                )
                time.sleep(wait_time)

        raise RuntimeError(
            f"Failed after {max_retries} retries. Last error: {last_error}"
        )

    # Read already processed question IDs if results CSV exists
    processed_ids = set()

    if output_csv.exists():
        old_results = pd.read_csv(output_csv, encoding="utf-8-sig")

        if "question_id" in old_results.columns:
            processed_ids = set(old_results["question_id"].astype(str))
        elif "Question_id" in old_results.columns:
            processed_ids = set(old_results["Question_id"].astype(str))

        for col in output_columns:
            if col not in old_results.columns:
                old_results[col] = ""

        old_results = old_results[output_columns]
        results = old_results.to_dict("records")
    else:
        old_results = pd.DataFrame(columns=output_columns)
        results = []

    print(f"Already processed: {len(processed_ids)} questions")

    newly_processed = 0

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        question_id = (
            clean_value(row[question_id_col])
            if question_id_col is not None
            else str(idx)
        )

        if str(question_id) in processed_ids:
            continue

        gold = normalize_gold(row[gold_col])

        group = clean_value(row["Group"]).upper() if "Group" in df.columns else "ABCDE"

        allowed_letters = [
            letter for letter in group
            if letter in ["A", "B", "C", "D", "E","F"]
        ]

        if not allowed_letters:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        prompt = clean_value(row[prompt_col])
        prompt += (
            "\n\nReturn exactly one uppercase letter from the allowed options. "
            "No explanation."
        )

        call_result = call_model(
            prompt=prompt,
            model=model,
            max_retries=max_retries,
        )

        raw_output = call_result["raw_output"]
        stop_reason = call_result["stop_reason"]
        refusal_category = call_result["refusal_category"]
        refusal_type = call_result["refusal_type"]

        if stop_reason == "refusal":
            prediction = "REFUSAL"
            correct = 0
        else:
            prediction = extract_prediction(raw_output, allowed_letters)
            correct = int(prediction == gold)

        result = {
            "question_id": question_id,
            "gold": gold,
            "prediction": prediction,
            "raw_output": raw_output,
            "correct": correct,
            "level": clean_value(row[level_col]) if level_col in df.columns else "",
            "medical_specialty": (
                clean_value(row[medical_specialty_col])
                if medical_specialty_col in df.columns
                else ""
            ),
            "model": model,
            "stop_reason": stop_reason,
            "refusal_category": refusal_category,
            "refusal_type": refusal_type,
        }

        results.append(result)
        processed_ids.add(str(question_id))
        newly_processed += 1

        if save_every and newly_processed % save_every == 0:
            pd.DataFrame(results, columns=output_columns).to_csv(
                output_csv,
                index=False,
                encoding="utf-8-sig",
            )

    results_df = pd.DataFrame(results, columns=output_columns)

    results_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return results_df

In [ ]:
# results = run_claude_opus48_on_questions(
#     input_csv="data/cleaned/medarabench_test_with_prompts.csv",
#     output_csv="results/claude_opus48_results.csv",
#     prompt_col="Prompt",
#     model="claude-opus-4-8",
#     gold_col="Correct Answer",
#     question_id_col="Question Number",
#     level_col="Level",
#     medical_specialty_col="Medical Specialty",
# )

Already processed: 25 questions


100%|██████████| 25/25 [00:00<00:00, 11931.91it/s]
